In [1]:
# CELL 1: IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os, joblib
from scipy import stats
from scipy.special import rel_entr
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb

os.makedirs('../results', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

print('=' * 60)
print('   NOTEBOOK 06: MODEL MONITORING & DRIFT DETECTION')
print('=' * 60)
print('''
Context:
  Fraud patterns evolve constantly — fraudsters adapt.
  A model trained on January data may lose 10-15% AUC by April.
  This notebook shows HOW to detect drift and WHEN to retrain.

Key metrics used:
  PSI  → Population Stability Index (feature drift)
  KL   → KL Divergence (prediction score drift)
  AUC  → Rolling AUC-ROC over simulated time windows
''')

   NOTEBOOK 06: MODEL MONITORING & DRIFT DETECTION

Context:
  Fraud patterns evolve constantly — fraudsters adapt.
  A model trained on January data may lose 10-15% AUC by April.
  This notebook shows HOW to detect drift and WHEN to retrain.

Key metrics used:
  PSI  → Population Stability Index (feature drift)
  KL   → KL Divergence (prediction score drift)
  AUC  → Rolling AUC-ROC over simulated time windows



In [2]:
# CELL 2: LOAD MODEL + BASELINE DATA

X_train = np.load('../data/X_train.npy')
y_train = np.load('../data/y_train.npy')
X_test  = np.load('../data/X_test.npy')
y_test  = np.load('../data/y_test.npy')
feature_names = pd.read_csv('../data/feature_names.csv')['feature'].tolist()
model = joblib.load('../results/model_xgb.pkl')

# Baseline score
baseline_probs = model.predict_proba(X_test)[:, 1]
baseline_auc   = roc_auc_score(y_test, baseline_probs)

print(f'Model loaded: XGBoost ({model.n_estimators} trees)')
print(f'X_test shape: {X_test.shape}')
print(f'\nBaseline (Month 1) AUC-ROC: {baseline_auc:.4f}  ← This is our starting point')
print(f'Goal: detect when this number degrades significantly')

Model loaded: XGBoost (500 trees)
X_test shape: (56962, 32)

Baseline (Month 1) AUC-ROC: 0.9782  ← This is our starting point
Goal: detect when this number degrades significantly


In [3]:
# CELL 3: SIMULATE DRIFT
# In production you'd use real new data each month.
# Here we simulate drift by progressively corrupting
# the test distribution — mimicking how fraud patterns
# change over time:
#   Month 1: baseline (no drift)
#   Month 2: slight drift (new fraud pattern emerges)
#   Month 3: moderate drift (fraud adapts to model)
#   Month 4: severe drift (model is significantly stale)

print('=== Simulating Concept Drift Across 6 Months ===')
print('Month 1 = original test set (no drift)')
print('Months 2-6 = progressively corrupted data')

rng = np.random.default_rng(42)
n   = len(X_test)
n_feat = X_test.shape[1]

def simulate_drift(X, y, drift_level=0.0, fraud_pattern_shift=0.0):
    """
    Simulate data drift at a given severity level.
    drift_level       : feature distribution shift (0=none, 1=severe)
    fraud_pattern_shift: change in fraud feature values (fraudsters adapting)
    """
    X_drifted = X.copy()
    # Gradual feature drift: add noise + mean shift to all features
    noise = rng.normal(0, drift_level * 0.5, X.shape)
    X_drifted += noise
    # Fraud pattern shift: change how fraud looks in top features (indices 0-5)
    fraud_mask = y == 1
    X_drifted[fraud_mask, :6] += fraud_pattern_shift
    return X_drifted

# 6 months of drift
months = {
    'Month 1 (Baseline)':    (0.00, 0.0),
    'Month 2 (Slight Drift)': (0.10, 0.3),
    'Month 3 (Mild Drift)':   (0.25, 0.8),
    'Month 4 (Moderate)':     (0.45, 1.5),
    'Month 5 (Significant)':  (0.70, 2.5),
    'Month 6 (Severe)':       (1.00, 4.0),
}

monthly_results = []
monthly_data    = {}

for month_name, (drift, fraud_shift) in months.items():
    X_m = simulate_drift(X_test, y_test, drift_level=drift, fraud_pattern_shift=fraud_shift)
    probs = model.predict_proba(X_m)[:, 1]
    auc   = roc_auc_score(y_test, probs)
    ap    = average_precision_score(y_test, probs)
    f1    = f1_score(y_test, (probs >= 0.3).astype(int))
    monthly_results.append({'Month': month_name, 'AUC-ROC': auc, 'Avg Prec': ap, 'F1': f1,
                             'Drift Level': drift, 'AUC Drop': baseline_auc - auc})
    monthly_data[month_name] = {'X': X_m, 'probs': probs}
    print(f'  {month_name:30s}: AUC={auc:.4f} (drop={baseline_auc-auc:+.4f})')

drift_df = pd.DataFrame(monthly_results)

=== Simulating Concept Drift Across 6 Months ===
Month 1 = original test set (no drift)
Months 2-6 = progressively corrupted data
  Month 1 (Baseline)            : AUC=0.9782 (drop=+0.0000)
  Month 2 (Slight Drift)        : AUC=0.9831 (drop=-0.0048)
  Month 3 (Mild Drift)          : AUC=0.9872 (drop=-0.0089)
  Month 4 (Moderate)            : AUC=0.9889 (drop=-0.0106)
  Month 5 (Significant)         : AUC=0.9779 (drop=+0.0004)
  Month 6 (Severe)              : AUC=0.9703 (drop=+0.0079)


In [4]:
# CELL 4: PSI CALCULATION
# PSI = Population Stability Index
#
# Standard thresholds:
#   PSI < 0.10  → Green  (no significant drift)
#   PSI 0.10-0.20 → Amber (investigate)
#   PSI > 0.20  → Red   (significant drift — retrain!)
#
# Formula: PSI = Σ (Actual% - Expected%) × ln(Actual% / Expected%)

print('=== PSI: Population Stability Index ===')
print('Measures: how much has the FEATURE distribution changed?')
print('Thresholds: PSI<0.10=OK | 0.10-0.20=Warn | >0.20=Retrain!')

def calculate_psi(baseline, current, n_bins=10):
    """Calculate PSI between baseline and current feature distributions."""
    # Use baseline quantiles as bin edges
    quantiles = np.linspace(0, 100, n_bins + 1)
    bins = np.percentile(baseline, quantiles)
    bins[0]  -= 1e-8
    bins[-1] += 1e-8

    base_pct = np.histogram(baseline, bins=bins)[0] / len(baseline)
    curr_pct = np.histogram(current,  bins=bins)[0] / len(current)

    # Clip to avoid log(0)
    base_pct = np.clip(base_pct, 1e-8, None)
    curr_pct = np.clip(curr_pct, 1e-8, None)

    psi = np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct))
    return psi

# Compute PSI for each month on prediction scores + top features
baseline_probs_ref = monthly_data['Month 1 (Baseline)']['probs']
baseline_X_ref     = monthly_data['Month 1 (Baseline)']['X']

psi_results = []
for month_name, data in monthly_data.items():
    psi_score  = calculate_psi(baseline_probs_ref, data['probs'])
    psi_v14    = calculate_psi(baseline_X_ref[:, 0], data['X'][:, 0])  # top feature
    alert = '🟢 OK' if psi_score < 0.10 else ('🟡 WARN' if psi_score < 0.20 else '🔴 RETRAIN!')
    psi_results.append({'Month': month_name, 'PSI_Scores': psi_score,
                        'PSI_V14': psi_v14, 'Alert': alert})
    print(f'  {month_name:30s}: PSI_scores={psi_score:.4f}  PSI_V14={psi_v14:.4f}  {alert}')

psi_df = pd.DataFrame(psi_results)

=== PSI: Population Stability Index ===
Measures: how much has the FEATURE distribution changed?
Thresholds: PSI<0.10=OK | 0.10-0.20=Warn | >0.20=Retrain!
  Month 1 (Baseline)            : PSI_scores=0.0000  PSI_V14=0.0000  🟢 OK
  Month 2 (Slight Drift)        : PSI_scores=0.0006  PSI_V14=0.0004  🟢 OK
  Month 3 (Mild Drift)          : PSI_scores=0.0017  PSI_V14=0.0063  🟢 OK
  Month 4 (Moderate)            : PSI_scores=0.0033  PSI_V14=0.0340  🟢 OK
  Month 5 (Significant)         : PSI_scores=0.0074  PSI_V14=0.0788  🟢 OK
  Month 6 (Severe)              : PSI_scores=0.0192  PSI_V14=0.1175  🟢 OK


In [5]:
# CELL 5: KL DIVERGENCE
# KL Divergence measures how much the prediction score
# distribution has shifted from baseline.
# If the model sees very different fraud patterns,
# the distribution of P(fraud) will change.

print('=== KL Divergence — Prediction Score Distribution Drift ===')
print('Measures: has the model\'s score distribution changed?')
print('High KL → model is seeing a very different input space')

def kl_divergence(p, q, n_bins=50):
    """KL(p || q) using histogram approximation."""
    min_val = min(p.min(), q.min())
    max_val = max(p.max(), q.max())
    bins = np.linspace(min_val, max_val, n_bins + 1)
    p_hist = np.histogram(p, bins=bins)[0].astype(float) + 1e-10
    q_hist = np.histogram(q, bins=bins)[0].astype(float) + 1e-10
    p_hist /= p_hist.sum()
    q_hist /= q_hist.sum()
    return np.sum(rel_entr(p_hist, q_hist))

kl_results = []
for month_name, data in monthly_data.items():
    kl = kl_divergence(baseline_probs_ref, data['probs'])
    kl_results.append({'Month': month_name, 'KL_Divergence': kl})
    flag = '' if kl < 0.05 else (' ⚠️' if kl < 0.20 else ' 🚨')
    print(f'  {month_name:30s}: KL = {kl:.5f}{flag}')

kl_df = pd.DataFrame(kl_results)
print('\nInterpretation: KL<0.05=stable | 0.05-0.20=monitor | >0.20=investigate')

=== KL Divergence — Prediction Score Distribution Drift ===
Measures: has the model's score distribution changed?
High KL → model is seeing a very different input space
  Month 1 (Baseline)            : KL = 0.00000
  Month 2 (Slight Drift)        : KL = 0.00079
  Month 3 (Mild Drift)          : KL = 0.00135
  Month 4 (Moderate)            : KL = 0.00218
  Month 5 (Significant)         : KL = 0.00405
  Month 6 (Severe)              : KL = 0.00402

Interpretation: KL<0.05=stable | 0.05-0.20=monitor | >0.20=investigate
